# 03 — Synthetic Feature Engineering

> ⚠️ **SYNTHETIC DATA — NOT REAL SURVEY RESPONSES**

**Input:**  `data/synthetic/processed/student_stress_synthetic_clean.csv`
**Outputs:**
- `data/synthetic/processed/X_features_synthetic.csv`
- `data/synthetic/processed/y_target_synthetic.csv`

**What this notebook does:**
1. Loads cleaned synthetic data (1000 × 38)
2. Drops leaky columns (stress questions + reversed + score)
3. Drops free-text column
4. Ordinal-encodes 11 ordered features
5. One-hot-encodes 4 nominal features
6. Saves feature matrix + target for model training

In [6]:
import os
import pandas as pd
import numpy as np

df = pd.read_csv("../data/synthetic/processed/student_stress_synthetic_clean.csv")
print("Loaded:", df.shape)

Loaded: (1000, 38)


## 1. Separate Features (X) and Target (y)

Exclude the leaky columns:
- All 10 stress questions
- The 4 reversed versions
- `stress_score` (used to derive target)
- `stress_level` (target itself)
- `biggest_stress_reason` (free text — reserved for future NLP)

In [7]:
y = df["stress_level"].copy()

leaky_cols = (
    [f"stress_q{i}" for i in range(1, 11)]
    + ["stress_q4_rev", "stress_q5_rev", "stress_q7_rev", "stress_q8_rev"]
    + ["stress_score", "stress_level"]
)
text_cols = ["biggest_stress_reason"]

X = df.drop(columns=leaky_cols + text_cols)

print("Target distribution:")
print(y.value_counts())
print()
print("X shape:", X.shape)
print("Feature count:", X.shape[1])
print()
print("Feature columns:")
for i, c in enumerate(X.columns, 1):
    print(f"  {i:2d}. {c}")

Target distribution:
stress_level
Moderate    342
Low         332
High        326
Name: count, dtype: int64

X shape: (1000, 21)
Feature count: 21

Feature columns:
   1. age
   2. gender
   3. university
   4. program
   5. year
   6. cgpa
   7. study_hours
   8. attendance
   9. assignment_stress
  10. exam_count
  11. sleep_hours
  12. social_media
  13. physical_activity
  14. part_time_job
  15. screen_time
  16. financial_stress
  17. social_support
  18. academic_life_satisfaction
  19. thought_break
  20. career_stress
  21. academic_performance


## 2. Ordinal Encoding

Encode the 11 ordered categorical features using the same mappings
as the real pipeline.

In [8]:
ordinal_mappings = {
    
    "age": {"17-19": 1, "20-22": 2, "23-25": 3, "Above 25": 4},
    "year": {"1st Year": 1, "2nd Year": 2, "3rd Year": 3, "4th Year": 4, "5th Year": 5},
    "cgpa": {"Below 2.0": 0, "Not Available Yet": 0, "2.0–2.49": 1, "2.5–2.99": 2, "3.0–3.49": 3, "3.5–4.0": 4},
    "study_hours": {"Less than 1": 1, "1–2": 2, "3–4": 3, "5–6": 4, "More than 6 hours": 5},
    "attendance": {"Below 60%": 1, "60–70%": 2, "71–80%": 3, "81–90%": 4, "Above 90%": 5},
    "exam_count": {"1-2": 1, "3-4": 2, "5-6": 3, "More than 6": 4},
    "sleep_hours": {"Less than 5": 1, "5–6": 2, "7–8": 3, "More than 8": 4},
    "social_media": {"Less than 1 hour": 1, "1–2 hours": 2, "3–4 hours": 3, "5–6 hours": 4, "More than 6 hours": 5},
    "physical_activity": {"Never": 0, "1–2 days/week": 1, "3–4 days/week": 2, "5–6 days/week": 3, "Daily": 4},
    "screen_time": {"Less than 3 hours": 1, "3–5 hours": 2, "6–8 hours": 3, "More than 8 hours": 4},
    "thought_break": {"Never": 1, "Rarely": 2, "Sometimes": 3, "Often": 4, "Very Often": 5},
}

X_encoded = X.copy()
for col, mapping in ordinal_mappings.items():
    X_encoded[col] = X_encoded[col].map(mapping)
    n_missing = X_encoded[col].isna().sum()
    if n_missing > 0:
        print(f"⚠️  {col}: {n_missing} unmapped → {X[col][X_encoded[col].isna()].unique()}")
    else:
        print(f"OK: {col} encoded")

print()
print("Shape after ordinal encoding:", X_encoded.shape)

OK: age encoded
OK: year encoded
OK: cgpa encoded
OK: study_hours encoded
OK: attendance encoded
OK: exam_count encoded
OK: sleep_hours encoded
OK: social_media encoded
OK: physical_activity encoded
OK: screen_time encoded
OK: thought_break encoded

Shape after ordinal encoding: (1000, 21)


## 3. One-Hot Encoding

Encode the 4 nominal features with `drop_first=True`.

In [9]:
nominal_cols = ["gender", "university", "program", "part_time_job"]

X_final = pd.get_dummies(
    X_encoded,
    columns=nominal_cols,
    drop_first=True,
    dtype=int,
)

print("Shape after one-hot:", X_final.shape)
print()
print("All feature columns:")
for i, c in enumerate(X_final.columns, 1):
    print(f"  {i:2d}. {c}")

Shape after one-hot: (1000, 38)

All feature columns:
   1. age
   2. year
   3. cgpa
   4. study_hours
   5. attendance
   6. assignment_stress
   7. exam_count
   8. sleep_hours
   9. social_media
  10. physical_activity
  11. screen_time
  12. financial_stress
  13. social_support
  14. academic_life_satisfaction
  15. thought_break
  16. career_stress
  17. academic_performance
  18. gender_Male
  19. gender_Prefer not to say
  20. university_Kathmandu University
  21. university_Madhyapaschim University
  22. university_Other
  23. university_Pokhara University
  24. university_Purbanchal University
  25. university_Rajarshi Janak University
  26. university_Sudurpaschim University
  27. university_Tribhuvan University
  28. program_BBS
  29. program_BCA
  30. program_BDS
  31. program_BE/B.Tech
  32. program_BHM
  33. program_BIM
  34. program_BSc CSIT
  35. program_BSc Nursing
  36. program_Law(LLB)
  37. program_MBBS
  38. part_time_job_Yes


## 4. Save Feature Matrix and Target

In [10]:
os.makedirs("../data/synthetic/processed", exist_ok=True)

X_final.to_csv("../data/synthetic/processed/X_features_synthetic.csv", index=False)
y.to_csv("../data/synthetic/processed/y_target_synthetic.csv", index=False)

print("Saved:")
print(f"  X_features_synthetic.csv  shape: {X_final.shape}")
print(f"  y_target_synthetic.csv    shape: {y.shape}")
print()
print("Target distribution:")
print(y.value_counts())

Saved:
  X_features_synthetic.csv  shape: (1000, 38)
  y_target_synthetic.csv    shape: (1000,)

Target distribution:
stress_level
Moderate    342
Low         332
High        326
Name: count, dtype: int64


## Summary

- ✅ Loaded cleaned synthetic data (1000 × 38)
- ✅ Dropped 17 leaky/text columns
- ✅ Kept 21 base features
- ✅ Ordinal-encoded 11 features
- ✅ One-hot-encoded 4 nominal features
- ✅ Saved X_features_synthetic.csv and y_target_synthetic.csv

**Next:** `04_synth_model_training.ipynb`